# NYC TLC taxi pickups: street-grid density without a basemap

NYC OpenData's 2015 TLC trip table retains pickup longitude and latitude.
The SODA API returns only those three requested columns, then
XY renders the same rows three ways: automatic scatter density,
explicit hexbin aggregation, and 24 hourly facets. Manhattan's street
grid emerges from pickup points alone.

The default reads one million valid pickups in 50,000-row pages.
Increase `TLC_MAX_ROWS` to scale the example; each page is cached so an
interrupted run resumes without repeating completed downloads.

**Source:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)
and the [2015 Yellow Taxi Trip Data table](https://data.cityofnewyork.us/d/2yzn-sicd).
TLC notes that trip records are published as submitted and may contain
inaccuracies.

Install beside XY with `python -m pip install numpy requests xy`.


In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "nyc-tlc"
DATA_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS = int(os.getenv("TLC_MAX_ROWS", "1000000"))
PAGE_SIZE = int(os.getenv("TLC_PAGE_SIZE", "50000"))
REQUEST_DELAY = float(os.getenv("TLC_REQUEST_DELAY", "0.1"))
if MAX_ROWS <= 0 or not 1 <= PAGE_SIZE <= 50_000 or REQUEST_DELAY < 0:
    raise ValueError("use positive rows, a page size <= 50,000, and a non-negative delay")

SESSION = requests.Session()
SESSION.headers["User-Agent"] = "xy-real-world-notebook/1.0"
app_token = os.getenv("SOCRATA_APP_TOKEN")
if app_token:
    SESSION.headers["X-App-Token"] = app_token
SESSION.mount(
    "https://",
    HTTPAdapter(
        max_retries=Retry(
            total=6,
            backoff_factor=1.0,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods={"GET"},
            respect_retry_after_header=True,
        )
    ),
)

API_URL = "https://data.cityofnewyork.us/resource/2yzn-sicd.csv"
parts = []
for offset in range(0, MAX_ROWS, PAGE_SIZE):
    limit = min(PAGE_SIZE, MAX_ROWS - offset)
    page_path = DATA_DIR / f"pickups-{offset:09d}-{limit:05d}.csv"
    if not page_path.exists():
        response = SESSION.get(
            API_URL,
            params={
                "$select": "pickup_longitude,pickup_latitude,pickup_datetime",
                "$where": (
                    "pickup_longitude between -74.10 and -73.70 "
                    "and pickup_latitude between 40.55 and 40.95"
                ),
                "$order": ":id",
                "$limit": limit,
                "$offset": offset,
            },
            stream=True,
            timeout=(30, 600),
        )
        response.raise_for_status()
        partial = page_path.with_suffix(".csv.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
        partial.replace(page_path)
        time.sleep(REQUEST_DELAY)

    page = np.loadtxt(
        page_path,
        delimiter=",",
        quotechar='"',
        skiprows=1,
        dtype=[("longitude", "f8"), ("latitude", "f8"), ("pickup_time", "U23")],
        encoding="utf-8",
        ndmin=1,
    )
    if page.size == 0:
        break
    parts.append(page)
    print(f"loaded {sum(part.size for part in parts):,} pickups")
    if page.size < limit:
        break

if not parts:
    raise RuntimeError("NYC OpenData returned no pickup rows")
pickups = np.concatenate(parts)
longitude = pickups["longitude"]
latitude = pickups["latitude"]
pickup_time = pickups["pickup_time"].astype("datetime64[ms]")
hour = (pickup_time.astype("datetime64[h]").astype(np.int64) % 24).astype(np.int16)
print(f"{longitude.size:,} valid pickup locations")

In [ ]:
density_chart = xy.scatter_chart(
    xy.scatter(
        longitude,
        latitude,
        color="#fbbf24",
        size=1.0,
        opacity=0.7,
        density=True,
    ),
    xy.x_axis(label="longitude", domain=(-74.05, -73.75)),
    xy.y_axis(label="latitude", domain=(40.58, 40.92)),
    xy.theme(
        background="#030712",
        text_color="#f8fafc",
        grid_color="#111827",
        axis_color="#94a3b8",
    ),
    title=f"NYC taxi pickup density · {longitude.size:,} trips",
    width=820,
    height=820,
)
print(
    "scatter tier:",
    density_chart.figure().build_payload()[0]["traces"][0]["tier"],
)
density_chart

In [ ]:
hexbin_chart = xy.hexbin_chart(
    xy.hexbin(
        longitude,
        latitude,
        gridsize=(180, 220),
        mincnt=2,
        bins="log",
        colormap="magma",
    ),
    xy.x_axis(label="longitude", domain=(-74.05, -73.75)),
    xy.y_axis(label="latitude", domain=(40.58, 40.92)),
    xy.colorbar(title="pickup count (log)"),
    xy.theme(
        background="#030712",
        text_color="#f8fafc",
        grid_color="#111827",
        axis_color="#94a3b8",
    ),
    title=f"NYC taxi pickup density · {longitude.size:,} trips · explicit hexbin",
    width=820,
    height=820,
)
hexbin_chart

In [ ]:
hourly_data = {
    "longitude": longitude,
    "latitude": latitude,
    "hour": hour,
}
hourly_chart = xy.facet_chart(
    xy.scatter(
        x="longitude",
        y="latitude",
        color="#38bdf8",
        size=1.0,
        opacity=0.7,
        density=True,
    ),
    xy.x_axis(label="longitude", domain=(-74.05, -73.75)),
    xy.y_axis(label="latitude", domain=(40.58, 40.92)),
    by="hour",
    data=hourly_data,
    cols=6,
    share_x=True,
    share_y=True,
    title="NYC taxi pickup density by local hour",
    width=1260,
    height=220,
    gap=8,
)
hourly_chart